## Earthkit Data - GRIB

In this notebook you will see how to:

- inspect GRIB data
- work with fields and fieldlists
- modify fields
- convert GRIB to Xarray
- convert GRIB to Pandas

## Getting the data

First we read some GRIB data from disk with ``from_source()``. The returned object provides some basic information but its primary goal is to convert the data into the required representation for further work. The actual data loading is deferred, as much as possible, until the data is converted into a given type.

In [1]:
import earthkit.data as ekd

d = ekd.from_source("file", "test4.grib")
d

path,test4.grib
size,509.5 KiB
types,"fieldlist, pandas, xarray, numpy, array"


In [2]:
d.available_types

['fieldlist', 'pandas', 'xarray', 'numpy', 'array']

## Fieldlists and fields

GRIB data can be converted into a FieldList, where each field represents a GRIB message. In earthkit a field is a horizontal slice of the atmosphere at a given time. In this sense the Field object is generic enough to be able to represent other types of data than GRIB (e.g. NetCDF, GeoTIFF, dictionary data etc.)

In [3]:
# fl is a fieldlist
fl = d.to_fieldlist()

In [4]:
len(fl)

4

In [5]:
# ls() lists the fields in the fieldlist
fl.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
2,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll
3,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


In [6]:
# we can iterate through the fields
for f in fl: 
    print(f)

Field(t, 2007-01-01 12:00:00, 2007-01-01 12:00:00, 0:00:00, 500, pressure, 0, regular_ll)
Field(z, 2007-01-01 12:00:00, 2007-01-01 12:00:00, 0:00:00, 500, pressure, 0, regular_ll)
Field(t, 2007-01-01 12:00:00, 2007-01-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)
Field(z, 2007-01-01 12:00:00, 2007-01-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)


#### Fields

In [7]:
# f is the first field in the fieldlist
f = fl[0]

A Field is composed of format independent components, each containing a set of metadata keys. The most important components and keys are as follows:

- parameter:
    - parameter.variable
    - parameter.units
    - parameter.chem_name
- time:
    - time.base_datetime
    - time.valid_datetime
    - time.step
- vertical:
    - vertical.level
    - vertical.level_type
- geography:
    - geography.latitudes
    - geography.longitudes
    - geography.shape
- ensemble:
    - ensemble.member

There are two ways to access the related values from the components:

In [8]:
# use the get() method
f.get("time.base_datetime")

datetime.datetime(2007, 1, 1, 12, 0)

In [9]:
# use the key method on the component
f.time.base_datetime()

datetime.datetime(2007, 1, 1, 12, 0)

The components contain format independent metadata. This is most obvious when looking at the level type or the units.

In [10]:
# the GRIB metadata would be isobaricInhPa
f.vertical.level_type()

'pressure'

In [11]:
# the GRIB metadata would be K
f.parameter.units()

<Unit('kelvin')>

To get an overview about the components/keys of a field simply use the automatic display.

In [12]:
# this is the same as f.describe()
f

number_of_values,65160
array_type,ndarray
array_dtype,float64
variable,t
units,kelvin
valid_datetime,2007-01-01 12:00:00
base_datetime,2007-01-01 12:00:00
step,0:00:00
level,500
layer,None
level_type,pressure


#### Field values

In [13]:
f.to_numpy()

array([[228.04600525, 228.04600525, 228.04600525, ..., 228.04600525,
        228.04600525, 228.04600525],
       [228.60850525, 228.57920837, 228.5499115 , ..., 228.698349  ,
        228.66807556, 228.63877869],
       [228.33311462, 228.26768494, 228.20420837, ..., 228.53623962,
        228.46788025, 228.39952087],
       ...,
       [244.47471619, 244.41807556, 244.36045837, ..., 244.64268494,
        244.58702087, 244.53135681],
       [245.13487244, 245.1124115 , 245.088974  , ..., 245.20323181,
        245.18077087, 245.15830994],
       [246.16319275, 246.16319275, 246.16319275, ..., 246.16319275,
        246.16319275, 246.16319275]], shape=(181, 360))

#### Field geography

In [14]:
# the shape is based on the geography (when available)
f.shape

(181, 360)

We can use latlons() to get a tuple of the latitudes and longitudes arrays. Please note "geography.latlons" cannot be used as a key.

In [15]:
f.geography.latlons()

(array([[ 90.,  90.,  90., ...,  90.,  90.,  90.],
        [ 89.,  89.,  89., ...,  89.,  89.,  89.],
        [ 88.,  88.,  88., ...,  88.,  88.,  88.],
        ...,
        [-88., -88., -88., ..., -88., -88., -88.],
        [-89., -89., -89., ..., -89., -89., -89.],
        [-90., -90., -90., ..., -90., -90., -90.]], shape=(181, 360)),
 array([[  0.,   1.,   2., ..., 357., 358., 359.],
        [  0.,   1.,   2., ..., 357., 358., 359.],
        [  0.,   1.,   2., ..., 357., 358., 359.],
        ...,
        [  0.,   1.,   2., ..., 357., 358., 359.],
        [  0.,   1.,   2., ..., 357., 358., 359.],
        [  0.,   1.,   2., ..., 357., 358., 359.]], shape=(181, 360)))

## Modifying fields

In [16]:
# Set generates a new field with updated values/components
vals = f.values + 2.0
f1 = f.set({"vertical.level": 700, "values": vals})
f1.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,700,pressure,0,regular_ll


In [17]:
f.get("vertical.level"), f1.get("vertical.level")

(500, 700)

In [18]:
f.values.max(), f1.values.max()

(np.float64(273.33799743652344), np.float64(275.33799743652344))

## Fieldlist subsetting

In [19]:
fl.sel({"parameter.variable": "t"}).ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


## Arithmetics

In [20]:
# select geopotential fields
z = fl.sel({"parameter.variable": "z"})

Compute the geopotential height. The result will be a new fieldlist entirely stored in memory.

In [21]:
# compute the geopotential height
# h and z are fieldlists
h = z / 9.81

# check the results
for zv, hv in zip(z, h):
    print(f"z-max: {zv.values.max()} h-max: {hv.values.max()}")

z-max: 58345.609375 h-max: 5947.564666156983
z-max: 16578.98828125 h-max: 1690.0089991080529


The metadata in the new "h" fields are still the same as in the "z" fields. We can correct it manually.

In [22]:
h.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


In [23]:
h = h.set({"parameter.variable": "gh", "parameter.units": "gpm"})
h.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,gh,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,gh,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


## Converting to Xarray

In [24]:
d = ekd.from_source("file", "pl.grib")
fl = d.to_fieldlist()
fl.unique("parameter.variable", "time.base_datetime", "time.step", "vertical.level")

{'parameter.variable': ('t', 'r'),
 'time.base_datetime': (datetime.datetime(2024, 6, 3, 0, 0),
  datetime.datetime(2024, 6, 3, 12, 0),
  datetime.datetime(2024, 6, 4, 0, 0),
  datetime.datetime(2024, 6, 4, 12, 0)),
 'time.step': (datetime.timedelta(0), datetime.timedelta(seconds=21600)),
 'vertical.level': (700, 500)}

In [25]:
ds = fl.to_xarray()
ds

<xarray.Dataset> Size: 176kB
Dimensions:                  (forecast_reference_time: 4, step: 2, level: 2,
                              latitude: 19, longitude: 36)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 32B 202...
  * step                     (step) timedelta64[ns] 16B 00:00:00 06:00:00
  * level                    (level) int64 16B 500 700
  * latitude                 (latitude) float64 152B 90.0 80.0 ... -80.0 -90.0
  * longitude                (longitude) float64 288B 0.0 10.0 ... 340.0 350.0
Data variables:
    r                        (forecast_reference_time, step, level, latitude, longitude) float64 88kB ...
    t                        (forecast_reference_time, step, level, latitude, longitude) float64 88kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

In [26]:
ds = fl.to_xarray(time_dim_mode="valid_time")
ds

<xarray.Dataset> Size: 176kB
Dimensions:     (valid_time: 8, level: 2, latitude: 19, longitude: 36)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 64B 2024-06-03 ... 2024-06-04T18:...
  * level       (level) int64 16B 500 700
  * latitude    (latitude) float64 152B 90.0 80.0 70.0 ... -70.0 -80.0 -90.0
  * longitude   (longitude) float64 288B 0.0 10.0 20.0 ... 330.0 340.0 350.0
Data variables:
    r           (valid_time, level, latitude, longitude) float64 88kB ...
    t           (valid_time, level, latitude, longitude) float64 88kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

In [27]:
ds["t"].earthkit._reference_field

number_of_values,684
array_type,ndarray
array_dtype,float64
variable,t
units,kelvin
valid_datetime,2024-06-03 00:00:00
base_datetime,2024-06-03 00:00:00
step,0:00:00
level,500
layer,None
level_type,pressure


In [28]:
ds["t"].earthkit.grid_spec

{'grid': [10.0, 10.0], 'area': [90.0, 0.0, -90.0, 350.0]}

## Converting to Pandas

In [29]:
fl[0].to_pandas()

,lat,lon,value,datetime
0,90.0,0.0,265.464554,2024-06-03
1,90.0,10.0,265.464554,2024-06-03
2,90.0,20.0,265.464554,2024-06-03
3,90.0,30.0,265.464554,2024-06-03
4,90.0,40.0,265.464554,2024-06-03
...,...,...,...,...
679,-90.0,310.0,237.597366,2024-06-03
680,-90.0,320.0,237.597366,2024-06-03
681,-90.0,330.0,237.597366,2024-06-03
682,-90.0,340.0,237.597366,2024-06-03


In [30]:
import earthkit.data as ekd

d = ekd.from_source("file", "test4.grib")
d

path,test4.grib
size,509.5 KiB
types,"fieldlist, pandas, xarray, numpy, array"


In [31]:
d.available_types

['fieldlist', 'pandas', 'xarray', 'numpy', 'array']

In [32]:
d.to_fieldlist().ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
2,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll
3,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


In [33]:
d.to_xarray()

<xarray.Dataset> Size: 2MB
Dimensions:    (level: 2, latitude: 181, longitude: 360)
Coordinates:
  * level      (level) int64 16B 500 850
  * latitude   (latitude) float64 1kB 90.0 89.0 88.0 87.0 ... -88.0 -89.0 -90.0
  * longitude  (longitude) float64 3kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
Data variables:
    t          (level, latitude, longitude) float64 1MB ...
    z          (level, latitude, longitude) float64 1MB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF